# Denomination / Banknote Class Recognition

This notebook provides an original transfer-learning workflow using **EfficientNetB0**.

It is suitable for an image-folder dataset where each folder represents a verified class.  
For clean denomination recognition, use a clean Indian-currency dataset with denomination labels.


In [ ]:
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path("data/denomination")
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Create data/denomination/<class_name>/... or see data/README.md."
    )

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Classes:", class_names)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])


In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        patience=4,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        patience=2,
        factor=0.3
    ),
    keras.callbacks.ModelCheckpoint(
        "models/denomination_effnet.keras",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)


In [ ]:
hist = pd.DataFrame(history.history)

plt.figure(figsize=(8,4))
plt.plot(hist["accuracy"], label="train")
plt.plot(hist["val_accuracy"], label="validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()
plt.tight_layout()
plt.show()


## Next step

After obtaining a stable baseline, optionally unfreeze the last blocks of EfficientNetB0 and fine-tune at a lower learning rate.

Do not report the Kaggle reference notebook's accuracy as this project's result. Record only metrics obtained from this repository's own training run.
